# Application Scorecard: US Consumer Loans

**Purpose:** build an application credit scorecard from information available on the application date,
validate it out of time, and compare it with the lender's own risk grade.

| Item | Choice |
|---|---|
| Population | 36-month unsecured personal loans |
| Development sample | Loans issued 2012 to 2014 (all matured by the snapshot date) |
| Out-of-time test | Loans issued 2015 (matured by the end of 2018) |
| Stability check | Loans issued 2016 to 2018 (scores only, no outcome needed) |
| Bad | Charged off, default, or 31+ days past due |
| Good | Fully paid |
| Excluded | Loans still running (indeterminate) |
| Method | Optimal binning, Weight of Evidence, logistic regression, points scaling (PDO) |
| Benchmark | The lender's sub-grade on the same out-of-time loans |

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from optbinning import BinningProcess
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

DATA_PATH = Path("data/loan.csv")
IMAGES = Path("images")
IMAGES.mkdir(exist_ok=True)

# Score scaling: 600 points at 50:1 good:bad odds, 20 points to double the odds
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50

# Characteristics known on the application date (application form and credit report at application)
RAW_FEATURES = [
    "loan_amnt", "purpose", "addr_state",
    "annual_inc", "emp_length", "home_ownership", "verification_status", "dti",
    "revol_bal", "revol_util", "open_acc", "total_acc", "mort_acc",
    "delinq_2yrs", "pub_rec", "inq_last_6mths", "mths_since_last_delinq",
    "earliest_cr_line",
]
CATEGORICAL = ["purpose", "addr_state", "home_ownership", "verification_status"]
MIN_IV = 0.02

## 2. Data and bad definition

Only the columns needed are read. Excluded on purpose to avoid leakage: the lender's grade, sub-grade,
interest rate and installment (they encode the lender's own decision), and every field recorded after
origination (payments, recoveries, last payment and last credit pull dates). `sub_grade` is read only
as the benchmark.

In [ ]:
raw = pd.read_csv(
    DATA_PATH,
    usecols=["issue_d", "term", "loan_status", "sub_grade"] + RAW_FEATURES,
    low_memory=False,
)

def prepare(frame):
    """Parse dates and terms, and convert text fields to numeric characteristics."""
    out = frame.copy()
    out["issue_date"] = pd.to_datetime(out["issue_d"], format="%b-%Y")
    out["issue_year"] = out["issue_date"].dt.year
    out["term_months"] = out["term"].str.extract(r"(\d+)", expand=False).astype(int)
    out["emp_length_years"] = (
        out["emp_length"].str.replace("< 1 year", "0", regex=False)
        .str.extract(r"(\d+)", expand=False).astype(float)
    )
    out["revol_util"] = pd.to_numeric(
        out["revol_util"].astype(str).str.replace("%", "", regex=False), errors="coerce"
    )
    earliest = pd.to_datetime(out["earliest_cr_line"], format="%b-%Y", errors="coerce")
    out["credit_history_months"] = (
        (out["issue_date"].dt.year - earliest.dt.year) * 12
        + (out["issue_date"].dt.month - earliest.dt.month)
    )
    return out.drop(columns=["issue_d", "term", "emp_length", "earliest_cr_line"])

loans = prepare(raw)
print(f"{len(loans):,} loans, issued {loans['issue_year'].min()} to {loans['issue_year'].max()}")

In [ ]:
TARGET = {"Fully Paid": 0, "Charged Off": 1, "Default": 1, "Late (31-120 days)": 1}

mature = loans[(loans["term_months"] == 36) & loans["issue_year"].between(2012, 2015)].copy()
mature["target"] = mature["loan_status"].map(TARGET)
print(f"Indeterminate loans excluded: {mature['target'].isna().sum():,}")
mature = mature.dropna(subset=["target"]).astype({"target": int})

sample_summary = mature.groupby("issue_year")["target"].agg(loans="count", bads="sum")
sample_summary["bad_rate_%"] = (sample_summary["bads"] / sample_summary["loans"] * 100).round(1)
sample_summary

In [ ]:
MODEL_FEATURES = [
    "loan_amnt", "purpose", "addr_state", "annual_inc", "emp_length_years", "home_ownership",
    "verification_status", "dti", "revol_bal", "revol_util", "open_acc", "total_acc", "mort_acc",
    "delinq_2yrs", "pub_rec", "inq_last_6mths", "mths_since_last_delinq", "credit_history_months",
]

dev = mature[mature["issue_year"] <= 2014].copy()
oot = mature[mature["issue_year"] == 2015].copy()
print(f"Development: {len(dev):,} loans, bad rate {dev['target'].mean():.1%}")
print(f"Out-of-time: {len(oot):,} loans, bad rate {oot['target'].mean():.1%}")

(dev[MODEL_FEATURES].isna().mean() * 100).round(1).sort_values(ascending=False).head(5).rename("missing_%")

## 3. Binning, Weight of Evidence and Information Value

Each characteristic is grouped into at most 6 bins of at least 5% of loans. Missing values form their
own bin, so a blank such as "no delinquency on record" becomes information. Characteristics with
Information Value of at least 0.02 are kept.

In [ ]:
bp = BinningProcess(
    variable_names=MODEL_FEATURES,
    categorical_variables=CATEGORICAL,
    max_n_bins=6,
    min_bin_size=0.05,
)
bp.fit(dev[MODEL_FEATURES], dev["target"])

iv_table = (
    bp.summary()[["name", "dtype", "n_bins", "iv"]]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)
selected = iv_table.loc[iv_table["iv"] >= MIN_IV, "name"].tolist()
print("Selected:", selected)
iv_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#1F4E79" if iv >= MIN_IV else "#BFBFBF" for iv in iv_table["iv"]]
ax.barh(iv_table["name"], iv_table["iv"], color=colors)
ax.axvline(MIN_IV, color="#C0392B", linestyle="--", linewidth=1)
ax.text(MIN_IV, len(iv_table) - 0.5, " IV = 0.02 threshold", color="#C0392B", va="top")
ax.invert_yaxis()
ax.set_xlabel("Information Value")
ax.set_title("Information Value by characteristic (development sample)")
plt.tight_layout()
plt.savefig(IMAGES / "01_iv_ranking.png", dpi=150)
plt.show()

In [ ]:
dti_bins = bp.get_binned_variable("dti").binning_table.build().iloc[:-1]
dti_bins = dti_bins[dti_bins["Count"] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dti_bins["Bin"].astype(str), dti_bins["Event rate"] * 100, color="#1F4E79")
ax.set_ylabel("Bad rate (%)")
ax.set_xlabel("DTI bin")
ax.set_title("DTI: bad rate rises steadily across bins")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(IMAGES / "02_dti_binning.png", dpi=150)
plt.show()

dti_bins[["Bin", "Count", "Event rate", "WoE", "IV"]].round(4)

## 4. Logistic regression

The model is fitted on the WoE values of the selected characteristics. Checks: all coefficients share
the same sign (required for WoE inputs, so every characteristic moves risk in the intended direction),
all are significant, and no two characteristics are highly correlated.

In [ ]:
X_dev = bp.transform(dev[MODEL_FEATURES], metric="woe")[selected]
y_dev = dev["target"]

corr = X_dev.corr()
high_corr = corr.where(lambda c: (c.abs() > 0.5) & (c.abs() < 1)).stack().dropna()
print("Pairs with |correlation| > 0.5:", "none" if high_corr.empty else high_corr.to_dict())

model = sm.Logit(y_dev, sm.add_constant(X_dev)).fit(disp=False)
coefficients = model.summary2().tables[1][["Coef.", "P>|z|"]]
coefficients.round(4)

## 5. Validation

Gini and KS on the development sample and on the out-of-time sample, with the lender's sub-grade on the
same out-of-time loans as the benchmark.

In [ ]:
def gini_ks(y, p):
    auc = roc_auc_score(y, p)
    ks = ks_2samp(p[y == 1], p[y == 0]).statistic * 100
    return round(2 * auc - 1, 3), round(ks, 1)

p_dev = model.predict(sm.add_constant(X_dev)).values
X_oot = bp.transform(oot[MODEL_FEATURES], metric="woe")[selected]
p_oot = model.predict(sm.add_constant(X_oot)).values
grade_rank = oot["sub_grade"].rank(method="dense").values   # A1 = safest ... G5 = riskiest

validation = pd.DataFrame(
    [
        ("Scorecard, development (2012-2014)", *gini_ks(y_dev.values, p_dev)),
        ("Scorecard, out-of-time (2015)", *gini_ks(oot["target"].values, p_oot)),
        ("Lender's sub-grade, out-of-time (2015)", *gini_ks(oot["target"].values, grade_rank)),
    ],
    columns=["model", "gini", "ks"],
)
validation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(validation["model"], validation["gini"], color=["#5B9BD5", "#1F4E79", "#BFBFBF"])
for i, g in enumerate(validation["gini"]):
    ax.text(g + 0.005, i, f"{g:.3f}", va="center")
ax.invert_yaxis()
ax.set_xlabel("Gini")
ax.set_xlim(0, 0.45)
ax.set_title("Discrimination: scorecard vs the lender's grade")
plt.tight_layout()
plt.savefig(IMAGES / "03_gini_comparison.png", dpi=150)
plt.show()

## 6. Scaling to points

Scores use the standard PDO method: 600 points at 50:1 good:bad odds and 20 points to double the
odds. Each applicant's score is the sum of the points for their bin in each characteristic.

In [ ]:
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def to_score(p_bad):
    return offset + factor * np.log((1 - p_bad) / p_bad)

dev["score"] = to_score(p_dev)
oot["score"] = to_score(p_oot)

n, b0 = len(selected), model.params["const"]
rows = []
for feature in selected:
    bins = bp.get_binned_variable(feature).binning_table.build().iloc[:-1]
    for _, r in bins[bins["Count"] > 0].iterrows():
        points = offset / n - factor * (b0 / n + model.params[feature] * r["WoE"])
        rows.append({"characteristic": feature, "bin": str(r["Bin"]),
                     "loans": int(r["Count"]), "bad_rate_%": round(r["Event rate"] * 100, 1),
                     "points": round(points)})
points_table = pd.DataFrame(rows)

print(f"Score range (development): {dev['score'].min():.0f} to {dev['score'].max():.0f}, "
      f"mean {dev['score'].mean():.0f}")
points_table

## 7. Score bands and cut-off

Development scores define 10 equal-sized bands. Applied to the out-of-time loans, they show the bad rate
by band and the trade-off between approval rate and bad rate at each possible cut-off.

In [ ]:
edges = np.quantile(dev["score"], np.linspace(0, 1, 11))
edges[0], edges[-1] = -np.inf, np.inf
dev["band"] = pd.cut(dev["score"], bins=edges, labels=range(1, 11))
oot["band"] = pd.cut(oot["score"], bins=edges, labels=range(1, 11))

bands = oot.groupby("band", observed=True).agg(
    min_score=("score", "min"), loans=("target", "count"), bads=("target", "sum")
)
bands["bad_rate_%"] = (bands["bads"] / bands["loans"] * 100).round(1)
bands["min_score"] = bands["min_score"].round()
bands

In [ ]:
total_bads = oot["target"].sum()
rows = []
for k in range(1, 11):
    approved = oot[oot["band"].astype(int) >= k]
    rows.append({
        "decline_bands_below": k,
        "min_score_approved": int(bands.loc[k, "min_score"]),
        "approval_rate_%": round(len(approved) / len(oot) * 100, 1),
        "bad_rate_approved_%": round(approved["target"].mean() * 100, 1),
        "bads_avoided_%": round((1 - approved["target"].sum() / total_bads) * 100, 1),
    })
cutoffs = pd.DataFrame(rows)
cutoffs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bands.index.astype(str), bands["bad_rate_%"], color="#1F4E79")
for i, v in enumerate(bands["bad_rate_%"]):
    ax.text(i, v + 0.4, f"{v}%", ha="center")
ax.axhline(oot["target"].mean() * 100, color="#E07B39", linestyle="--", linewidth=1)
ax.set_xlabel("Score band (1 = lowest score)")
ax.set_ylabel("Bad rate (%)")
ax.set_title("Out-of-time bad rate by score band (2015)")
plt.tight_layout()
plt.savefig(IMAGES / "04_bad_rate_by_band.png", dpi=150)
plt.show()

## 8. Stability (PSI)

The score is applied to 36-month loans issued 2016 to 2018 and its band distribution is compared with
development. PSI below 0.10 is stable, 0.10 to 0.25 needs monitoring, above 0.25 is a significant shift.

In [ ]:
recent = loans[(loans["term_months"] == 36) & loans["issue_year"].between(2016, 2018)].copy()
X_recent = bp.transform(recent[MODEL_FEATURES], metric="woe")[selected]
recent["score"] = to_score(model.predict(sm.add_constant(X_recent)).values)
recent["band"] = pd.cut(recent["score"], bins=edges, labels=range(1, 11))

expected = dev["band"].value_counts(normalize=True).sort_index()

def psi(expected, actual):
    e, a = expected.clip(lower=0.0001), actual.clip(lower=0.0001)
    return float(((a - e) * np.log(a / e)).sum())

rows, shares = [], {"Development (2012-2014)": expected}
for year in [2016, 2017, 2018]:
    in_year = recent["issue_year"] == year
    actual = recent.loc[in_year, "band"].value_counts(normalize=True).sort_index()
    shares[str(year)] = actual
    value = psi(expected, actual)
    rows.append({"year": year, "loans": int(in_year.sum()),
                 "avg_score": round(recent.loc[in_year, "score"].mean(), 1),
                 "psi": round(value, 3),
                 "status": "Stable" if value < 0.10 else ("Monitor" if value < 0.25 else "Shift")})
stability = pd.DataFrame(rows)
stability

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, 11)
ax.bar(x - 0.2, shares["Development (2012-2014)"] * 100, width=0.4, color="#BFBFBF", label="Development (2012-2014)")
ax.bar(x + 0.2, shares["2018"] * 100, width=0.4, color="#1F4E79", label="2018")
ax.set_xticks(x)
ax.set_xlabel("Score band (1 = lowest score)")
ax.set_ylabel("Share of loans (%)")
ax.set_title("Score distribution: development vs 2018")
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES / "05_score_stability.png", dpi=150)
plt.show()

## 9. Conclusions

- **Discrimination:** Gini 0.259 on development and 0.254 out of time, so the scorecard generalises to
  new loans. It reaches about 70% of the lender's sub-grade (Gini 0.357) using 8 application
  characteristics and no credit bureau score.
- **Drivers:** income, revolving utilisation and recent credit inquiries carry the most points, followed
  by DTI and loan purpose.
- **Rank ordering:** the out-of-time bad rate falls from 25.9% in the lowest band to 6.6% in the highest.
- **Cut-off:** declining the lowest band (score below 526) avoids 14.9% of bad loans for an 8.6% fall in
  approvals, lowering the approved bad rate from 14.9% to 13.9%. Because this band's bad rate is close to
  grade E, which remained profitable, risk-based pricing or manual review is an alternative to a hard
  decline.
- **Stability:** PSI 0.024 (2016) and 0.043 (2017), rising to 0.101 in 2018 as the approved population
  moved toward safer borrowers. This would trigger a recalibration review, not a rebuild.

**Limitations:** the data holds approved loans only (no reject inference), has no credit bureau score,
and past-delinquency characteristics are weak because the lender had already screened most such
applicants out.